In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK setup
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords

# Sklearn for TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Load the news dataset
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')  # adjust filename as needed
print(df.shape)
print(df.dtypes)
df.head()

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Sample rows ===")
print(df.sample(5))

print("\n=== Columns ===")
print(df.columns.tolist())

In [ ]:
df['headline_length'] = df['headline'].astype(str).apply(len)
df['word_count'] = df['headline'].astype(str).apply(lambda x: len(x.split()))

print(df[['headline_length', 'word_count']].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['headline_length'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Headline Character Length Distribution')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Headline Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/raw/headline_length_dist.png', dpi=150)
plt.show()

In [ ]:
# Count articles per publisher
publisher_counts = df['publisher'].value_counts().head(20)

plt.figure(figsize=(12, 6))
publisher_counts.plot(kind='barh', color='teal', edgecolor='white')
plt.title('Top 20 Most Active Publishers')
plt.xlabel('Number of Articles')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../data/raw/publisher_analysis.png', dpi=150)
plt.show()

print(f"\nTotal unique publishers: {df['publisher'].nunique()}")

# If publisher names look like emails, extract domain
def extract_domain(pub):
    if '@' in str(pub):
        return str(pub).split('@')[-1]
    return str(pub)

df['publisher_domain'] = df['publisher'].apply(extract_domain)
print("\nTop domains:")
print(df['publisher_domain'].value_counts().head(10))

In [ ]:
# Parse dates (handle timezone info)
df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
df = df.dropna(subset=['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()
df['hour'] = df['date'].dt.hour
df['date_only'] = df['date'].dt.date

print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"\nArticles per year:\n{df['year'].value_counts().sort_index()}")

In [ ]:
daily_counts = df.groupby('date_only').size().reset_index(name='article_count')
daily_counts['date_only'] = pd.to_datetime(daily_counts['date_only'])
daily_counts = daily_counts.sort_values('date_only')

plt.figure(figsize=(16, 5))
plt.plot(daily_counts['date_only'], daily_counts['article_count'],
         linewidth=0.8, color='steelblue', alpha=0.8)
plt.fill_between(daily_counts['date_only'], daily_counts['article_count'],
                 alpha=0.2, color='steelblue')
plt.title('Daily Article Publication Frequency Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Articles')
plt.tight_layout()
plt.savefig('../data/raw/publication_frequency.png', dpi=150)
plt.show()

# Top spike days
print("Top 10 highest-volume days:")
print(daily_counts.nlargest(10, 'article_count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour of day
hour_counts = df['hour'].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values, color='mediumseagreen', edgecolor='white')
axes[0].set_title('Articles by Hour of Day (UTC-4)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(0, 24))

# Day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(day_counts.index, day_counts.values, color='mediumpurple', edgecolor='white')
axes[1].set_title('Articles by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/raw/publishing_time_analysis.png', dpi=150)
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_headline'] = df['headline'].apply(clean_text)

# TF-IDF top keywords
tfidf = TfidfVectorizer(max_features=50, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['clean_headline'])
feature_names = tfidf.get_feature_names_out()
tfidf_scores = tfidf_matrix.mean(axis=0).A1

tfidf_df = pd.DataFrame({'keyword': feature_names, 'score': tfidf_scores})
tfidf_df = tfidf_df.sort_values('score', ascending=False).head(20)

plt.figure(figsize=(12, 6))
plt.barh(tfidf_df['keyword'], tfidf_df['score'], color='darkorange', edgecolor='white')
plt.gca().invert_yaxis()
plt.title('Top 20 Keywords by TF-IDF Score')
plt.xlabel('Mean TF-IDF Score')
plt.tight_layout()
plt.savefig('../data/raw/tfidf_keywords.png', dpi=150)
plt.show()

In [ ]:
# CountVectorizer for LDA
cv = CountVectorizer(max_features=500, ngram_range=(1, 2), min_df=5)
cv_matrix = cv.fit_transform(df['clean_headline'])
cv_features = cv.get_feature_names_out()

# LDA with 5 topics
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(cv_matrix)

print("=== LDA Topics ===")
for i, topic in enumerate(lda.components_):
    top_words = [cv_features[j] for j in topic.argsort()[-10:]][::-1]
    print(f"Topic {i+1}: {', '.join(top_words)}")

In [ ]:
# Articles per stock ticker
stock_counts = df['stock'].value_counts().head(15)

plt.figure(figsize=(12, 5))
stock_counts.plot(kind='bar', color='cornflowerblue', edgecolor='white')
plt.title('Top 15 Most Covered Stocks')
plt.xlabel('Stock Symbol')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/raw/stock_coverage.png', dpi=150)
plt.show()